# Experiment 2b: Hyperparameter search + Rank ablation

## Cell 1: Install

In [ ]:
!pip install -q transformers peft datasets accelerate trl nbformat
!pip install --upgrade Pillow

## Cell 2: Load Muon

In [ ]:
%run /home/ubuntu/thesis-storage-1/muon.ipynb #Replace with ur file path

## Cell 3: Imports

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
from torch.utils.data import DataLoader
import copy
import os
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## Cell 4: Config

In [ ]:
MODEL_NAME   = 'microsoft/Phi-4-mini-instruct' #Replace it with your model name
LORA_RANKS   = [4, 8, 16, 32, 64]
LORA_DROPOUT = 0.05

# We attach LoRA adapters to all attention and MLP projection layers.
# This covers query, key, value, output projections in attention, and gate, up, down projections in the feed-forward block
LORA_TARGETS = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 
                'gate_proj', 'up_proj', 'down_proj']

ADAMW_LRS  = [1e-4, 3e-4, 1e-3] # Fine-tuning Optimal LR's (You can put your range in here)
MUON_LRS   = [5e-4, 2e-3, 5e-3] # Muon LR range is shifted higher (5e-4 to 5e-3) since Muon's RMS scaling means it typically needs larger learning rates than AdamW to match update


#Muon Hyperparams borowed form Kellar Jordan, Moonlightpaper, or standard nums
MUON_MOMENTUM     = 0.95
MUON_WD           = 0.1
MUON_UPDATE_SCALE = 0.3
ADAMW_WD          = 0.1

SEARCH_STEPS = 500
FULL_STEPS   = 7989
BATCH_SIZE   = 8
MAX_SEQ_LEN  = 512
GRAD_ACCUM   = 8

SAVE_DIR = '/home/ubuntu/thesis-storage-1/experiment_2b_results' #Replace with ur file path
os.makedirs(SAVE_DIR, exist_ok=True)

print('Config set.')

## Cell 5: Load Model and Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=False,
)

# Move to CPU immediately and keep there 
base_cpu = base_model.cpu()
torch.cuda.empty_cache()

print('Parameters:', round(sum(p.numel() for p in base_cpu.parameters()) / 1e9, 2), 'B')
dataset = load_dataset('zwhe99/commonsense_170k', split='train')
print('Dataset size:', len(dataset))

def tokenize(example):
    text = f"### Instruction:\n{example['instruction']}\n### Response:\n{example['output']}"
    tokens = tokenizer(
        text, truncation=True, max_length=MAX_SEQ_LEN,
        padding='max_length', return_tensors='pt',
    )
    tokens['labels'] = tokens['input_ids'].clone()
    return {k: v.squeeze(0) for k, v in tokens.items()}

tokenized = dataset.map(tokenize, remove_columns=dataset.column_names)
tokenized.set_format('torch')

## Cell 6: Train Function

In [ ]:
def run_training(model, dataloader, optimizer_type, lr, max_steps, grad_accum=GRAD_ACCUM):
    if optimizer_type == 'muon':
        muon_params, adamw_params = get_muon_and_adamw_params(model)
        optimizer = [Muon(muon_params, lr=lr, momentum=MUON_MOMENTUM,
                         weight_decay=MUON_WD, update_scale=MUON_UPDATE_SCALE)]
        if len(adamw_params) > 0:
            optimizer.append(torch.optim.AdamW(adamw_params, lr=lr, weight_decay=ADAMW_WD))
    else:
        optimizer = [torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=ADAMW_WD)]

    loss_history = []
    model.train()
    step = 0
    data_iter = iter(dataloader)

    while step < max_steps:
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            batch = next(data_iter)

        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss / grad_accum
        loss.backward()

        if (step + 1) % grad_accum == 0:
            for opt in optimizer:
                opt.step()
                opt.zero_grad()

        loss_history.append(loss.item() * grad_accum)
        step += 1

    final_loss = float(np.mean(loss_history[-50:]))
    return loss_history, final_loss

## Cell 7: LR Search

In [ ]:
search_results = {}
best_lrs = {}

for rank in LORA_RANKS:
    print(f'\n========== RANK {rank} LR SEARCH ==========')
    search_results[rank] = {'adamw': {}, 'muon': {}}

    for opt_type, lr_list in [('adamw', ADAMW_LRS), ('muon', MUON_LRS)]:
        best_lr   = None
        best_loss = float('inf')

        for lr in lr_list:
            # deepcopy from base_cpu
            torch.cuda.empty_cache()

            lora_config = LoraConfig(
                r=rank, lora_alpha=2*rank, lora_dropout=LORA_DROPOUT,
                target_modules=LORA_TARGETS, task_type=TaskType.CAUSAL_LM, bias='none',
            )
            model = get_peft_model(copy.deepcopy(base_cpu), lora_config)
            model = model.cuda()

            dataloader = DataLoader(tokenized, batch_size=BATCH_SIZE, shuffle=True)
            _, final_loss = run_training(model, dataloader, opt_type, lr, SEARCH_STEPS)

            print(f'  {opt_type.upper()} lr={lr:.0e} -> final_loss={final_loss:.4f}')
            search_results[rank][opt_type][str(lr)] = final_loss

            if final_loss < best_loss:
                best_loss = final_loss
                best_lr   = lr

            del model
            torch.cuda.empty_cache()

        if rank not in best_lrs:
            best_lrs[rank] = {}
        best_lrs[rank][opt_type] = best_lr
        print(f'  Best {opt_type.upper()} LR for rank {rank}: {best_lr:.0e}')

with open(os.path.join(SAVE_DIR, 'lr_search_results.json'), 'w') as f:
    json.dump({'search': {str(k): v for k,v in search_results.items()},
               'best_lrs': {str(k): v for k,v in best_lrs.items()}}, f, indent=2)


## Cell 8: Full Model Run

In [ ]:
all_results = {}

for rank in LORA_RANKS:
    print(f'\n========== RANK {rank} FULL RUN ==========')
    rank_results = {}

    for opt_type in ['adamw', 'muon']:
        best_lr = best_lrs[rank][opt_type]
        print(f'  {opt_type.upper()} using LR={best_lr:.0e}')

        # deepcopy from base_cpu
        torch.cuda.empty_cache()

        lora_config = LoraConfig(
            r=rank, lora_alpha=2*rank, lora_dropout=LORA_DROPOUT,
            target_modules=LORA_TARGETS, task_type=TaskType.CAUSAL_LM, bias='none',
        )
        model = get_peft_model(copy.deepcopy(base_cpu), lora_config)
        model = model.cuda()
        model.print_trainable_parameters()

        dataloader = DataLoader(tokenized, batch_size=BATCH_SIZE, shuffle=True)
        losses, final_loss = run_training(model, dataloader, opt_type, best_lr, FULL_STEPS)

        rank_results[opt_type] = {
            'losses': losses,
            'final_loss': final_loss,
            'best_lr': best_lr
        }

        torch.save(rank_results[opt_type],
                   os.path.join(SAVE_DIR, f'rank{rank}_{opt_type}_results.pt'))
        print(f'  Done. Final loss: {final_loss:.4f}. Saved.')

        del model
        torch.cuda.empty_cache()

    all_results[rank] = rank_results

## Cell 9: Plot 

In [ ]:
def smooth(vals, window=100):
    return np.convolve(vals, np.ones(window)/window, mode='valid')

fig, axes = plt.subplots(1, len(LORA_RANKS), figsize=(20, 4), sharey=False)

for i, rank in enumerate(LORA_RANKS):
    ax = axes[i]
    adamw_lr = all_results[rank]['adamw']['best_lr']
    muon_lr  = all_results[rank]['muon']['best_lr']

    adamw_smooth = smooth(all_results[rank]['adamw']['losses'])
    muon_smooth  = smooth(all_results[rank]['muon']['losses'])

    ax.plot(adamw_smooth, label=f'AdamW lr={adamw_lr:.0e}', color='blue', alpha=0.8)
    ax.plot(muon_smooth,  label=f'Muon lr={muon_lr:.0e}',  color='orange', alpha=0.8)
    ax.set_title(f'Rank {rank}')
    ax.set_xlabel('Step')
    if i == 0:
        ax.set_ylabel('Loss')
    ax.legend(fontsize=6)
    ax.grid(True, alpha=0.3)

plt.suptitle('Muon vs AdamW: Tuned LR per Rank (Phi-4-mini, LoRA)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'tuned_loss_curves.png'), dpi=150)
plt.show()

print('='*65)
print('EXPERIMENT 2b SUMMARY')
print('='*65)
print(f'{"Rank":<6} {"AdamW LR":<12} {"AdamW Loss":<14} {"Muon LR":<12} {"Muon Loss":<12} {"Winner"}')
print('-'*65)
for rank in LORA_RANKS:
    a   = all_results[rank]['adamw']['final_loss']
    m   = all_results[rank]['muon']['final_loss']
    alr = all_results[rank]['adamw']['best_lr']
    mlr = all_results[rank]['muon']['best_lr']
    winner = 'Muon' if m < a else 'AdamW'
    print(f'{rank:<6} {alr:<12.0e} {a:<14.4f} {mlr:<12.0e} {m:<12.4f} {winner}')